In [ ]:
corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

### BPE merging rules

In [ ]:
# 1.  ("hug", 10), ("pug", 5), ("pun", 12), ("bun", 4), ("hugs", 5)
# 2.  ("h" "u" "g", 10), ("p" "u" "g", 5), ("p" "u" "n", 12), ("b" "u" "n", 4), ("h" "u" "g" "s", 5)
# 3.  Vocabulary: ["b", "g", "h", "n", "p", "s", "u", "ug"]
#     Corpus: ("h" "ug", 10), ("p" "ug", 5), ("p" "u" "n", 12), ("b" "u" "n", 4), ("h" "ug" "s", 5)

#     Vocabulary: ["b", "g", "h", "n", "p", "s", "u", "ug", "un"]
#     Corpus: ("h" "ug", 10), ("p" "ug", 5), ("p" "un", 12), ("b" "un", 4), ("h" "ug" "s", 5)

### Functions

In [ ]:
from collections import defaultdict

def get_word_freqs(corpus):
    word_freqs = defaultdict(int)
    for text in corpus:
        words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
        new_words = [word for word, offset in words_with_offsets]
        for word in new_words:
            word_freqs[word] += 1
    return word_freqs

def get_vocab(word_freqs):
    vocab = set()
    for word in word_freqs:
        for c in word:
            vocab.add(c)
    vocab = sorted(list(vocab))
    vocab += ['<|endoftext|>']
    return vocab

def get_splits(word_freqs):
    splits = {word: [c for c in word] for word in word_freqs.keys()}
    return splits

def compute_pair_freqs(word_freqs,splits):
    pair_freqs = defaultdict(int)
    for word, freq in word_freqs.items():
        split = splits[word]
        if len(split) <= 1:
            continue
        else:
            for i in range(len(split)-1):
                left,right = split[i],split[i+1]
                pair_freqs[(left,right)] += freq
    return pair_freqs

def get_most_frequent_pair(pair_freqs):
    best_pair = None
    max_freq = -1
    for pair, freq in pair_freqs.items():
        if freq > max_freq:
            best_pair = pair
            max_freq = freq
    return best_pair


def merge_pairs(a,b,splits,word_freqs):
    for word in word_freqs:
        split = splits[word]
        if len(split) == 1:
            continue
        i = 0
        while i < len(split)-1:
            if split[i] == a and split[i+1]==b:
                split = split[:i] + [a+b] + split[i+2:]
            else:
                i+=1
        splits[word] = split
    return splits
    

### Construct `merges` log and final `vocab` until `max_vocab_size` achieved

In [23]:
from collections import defaultdict

word_freqs = get_word_freqs(corpus)
vocab = get_vocab(word_freqs)
splits = get_splits(word_freqs)
merges = {}

vocab_size = 50

while len(vocab) < vocab_size:
    pair_freqs = compute_pair_freqs(word_freqs, splits)
    a,b = get_most_frequent_pair(pair_freqs)
    splits = merge_pairs(a,b,splits,word_freqs)

    merges[(a,b)] = a+b
    vocab.append(a+b)

### Use `merges` final rule to construct new sentences

In [ ]:
def tokenize(text):
    # split the sentence using pretokenizer
    pre_tokenize_result = tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(text)
    # only grab the words
    pre_tokenized_text = [word for word, offset in pre_tokenize_result]
    # define the character for each words
    splits = [[l for l in word] for word in pre_tokenized_text]

    # refer back to the vocab merging rule, remember we need to scan the merge rule same with the order when they are created, so we put them in outer loop
    for pair, merge in merges.items():
        # for each splitted, do the character merging rule
        for idx, split in enumerate(splits):
            i = 0
            # if the specific index match with the observed pair in the merign rule, merge them
            # in a sliding window approach
            while i < len(split) - 1:
                if split[i] == pair[0] and split[i + 1] == pair[1]:
                    split = split[:i] + [merge] + split[i + 2 :]
                else:
                    i += 1
            # replace the splits for specific word
            splits[idx] = split

    # flatten each word arrays into one 1D array
    return sum(splits, [])

In [25]:
tokenize("This is not a token.")

['This', 'Ġis', 'Ġ', 'n', 'o', 't', 'Ġa', 'Ġtoken', '.']